In [43]:
import os
import pandas as pd
import json
from pathlib import Path
import subprocess
import base64

# Destination folder
ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
MQTT_PARTS_GZ_FOLDER = ROOT / "Data"/ "Landing_Zone" / "emf-mqtt-dump"

print(MQTT_PARTS_GZ_FOLDER.resolve())

F:\Users\Graham\Desktop\Github Repos\Met Police\EMF_Camp_DataEng\Data\Landing_Zone\emf-mqtt-dump


## combine and read files. 
Need to check this soltution has no effects on data integrety. 

In [44]:
gzip_files = sorted(
    MQTT_PARTS_GZ_FOLDER.glob("mqtt.emf.camp.csv.gz.*"),
    key=lambda path: int(path.name.rsplit(".", 1)[-1])
)

if len(gzip_files) != 2:
    raise ValueError(f"Expected 2 gzip parts, found {len(gzip_files)}: {gzip_files}")

combined_gzip = MQTT_PARTS_GZ_FOLDER / "combined_csv.gz"

with combined_gzip.open("wb") as output:
    for file in gzip_files:
        with file.open("rb") as part:
            while chunk := part.read(1024 * 1024):
                output.write(chunk)

columns = ["msg.topic", "payload", "msg.qos", "msg.retain", "time_now", "time_delta"]
df = pd.read_csv(combined_gzip, compression="gzip", header=None, names=columns)

print(f"Combined GZIP saved to: {combined_gzip}")

df.head()


Combined GZIP saved to: F:\Users\Graham\Desktop\Github Repos\Met Police\EMF_Camp_DataEng\Data\Landing_Zone\emf-mqtt-dump\combined_csv.gz


,msg.topic,payload,msg.qos,msg.retain,time_now,time_delta
0,open/jungle-breaks-served,NDE=,0,False,1.784160e+09,1.699473
1,open/wnstlyktrasdi,MDExMDExMTE=,0,False,1.784160e+09,1.627660
2,open/wnstlyktrasdi,MDExMTAxMDE=,0,False,1.784160e+09,4.000714
3,open/view-the-full-schedule-timeline-at,aHR0cHM6Ly9oZWFsdy5pbmc=,0,False,1.784160e+09,0.781992
4,open/hackers,eyJTY2VuZS1sb2NhdGlvbiI6ICJJTlQuIFBMQUdVRSdTIE...,0,False,1.784160e+09,0.005430


## Decode Payload and time

In [45]:
# Decode Base64 and parse JSON payloads into dictionaries
def decode_payload(value):
    if pd.isna(value):
        return {}

    try:
        decoded = base64.b64decode(value).decode("utf-8")
        payload = json.loads(decoded)
        return payload if isinstance(payload, dict) else {}
    except (ValueError, json.JSONDecodeError, UnicodeDecodeError, base64.binascii.Error):
        return {}

# Decode Payload
payloads = df["payload"].map(decode_payload)
df["payload"] = payloads

# Convert Unix time 
df["time_now"] = pd.to_datetime(
    df["time_now"], unit="s", utc=True, errors="coerce"
).dt.tz_convert("Europe/London")


display(df)

,msg.topic,payload,msg.qos,msg.retain,time_now,time_delta
0,open/jungle-breaks-served,{},0,False,2026-07-16 01:00:01.638832092+01:00,1.699473
1,open/wnstlyktrasdi,{},0,False,2026-07-16 01:00:03.266491890+01:00,1.627660
2,open/wnstlyktrasdi,{},0,False,2026-07-16 01:00:07.267206192+01:00,4.000714
3,open/view-the-full-schedule-timeline-at,{},0,False,2026-07-16 01:00:08.049198389+01:00,0.781992
4,open/hackers,"{'Scene-location': 'INT. PLAGUE'S LOFT.', 'Sce...",0,False,2026-07-16 01:00:08.054627895+01:00,0.005430
...,...,...,...,...,...,...
1927245,power/generator/s,"{'gencomm_version': 2.0, 'password_status': 3....",0,False,2026-07-20 12:59:57.016425133+01:00,1.005037
1927246,power/generator/s,"{'gencomm_version': 2.0, 'password_status': 3....",0,False,2026-07-20 12:59:58.011878967+01:00,0.995454
1927247,weather/hq,"{'runtime': 878378.0, 'heap': 139448.0, 'tempi...",0,False,2026-07-20 12:59:58.341118813+01:00,0.329240
1927248,power/generator/s,"{'gencomm_version': 2.0, 'password_status': 3....",0,False,2026-07-20 12:59:59.011588335+01:00,0.670470


In [46]:
print("Smallest time_now:", df["time_now"].min())
print("Largest time_now:", df["time_now"].max())
print(df.iloc[4]["payload"])

Smallest time_now: 2026-07-16 01:00:01.638832092+01:00
Largest time_now: 2026-07-20 12:59:59.755843163+01:00
{'Scene-location': "INT. PLAGUE'S LOFT.", 'Scene-description': 'Margo starts to protest.', 'Character': 'PLAGUE', 'Lines': 'Look, there is no right and wrong. There is only fun and boring. A thirty year prison sentence sounds pretty dull to me. Who do you prefer serves it, us? Or them?'}


In [47]:
matches = df[df["msg.topic"].str.contains("schedule", case=False, na=False)] # Might be good to santity check results using activity from here ?
display(matches.head())
display(matches["msg.topic"].iloc[1])

,msg.topic,payload,msg.qos,msg.retain,time_now,time_delta
3,open/view-the-full-schedule-timeline-at,{},0,False,2026-07-16 01:00:08.049198389+01:00,0.781992
15,open/view-the-full-schedule-timeline-at,{},0,False,2026-07-16 01:00:21.421551466+01:00,1.746532
28,open/view-the-full-schedule-timeline-at,{},0,False,2026-07-16 01:00:34.791072607+01:00,1.665159
41,open/view-the-full-schedule-timeline-at,{},0,False,2026-07-16 01:00:48.160285711+01:00,0.882986
65,open/view-the-full-schedule-timeline-at,{},0,False,2026-07-16 01:01:01.530912161+01:00,0.547135


'open/view-the-full-schedule-timeline-at'